# Projet 1 — Système expert : résolution des Tours de Hanoï

**Machine Learning — 2026/2027**

**Objectif :** concevoir un système expert à base de règles utilisant un moteur d'inférence par chaînage avant pour résoudre les Tours de Hanoï.


## 1. Objectif

Le problème consiste à déplacer les trois disques des Tours de Hanoï du pic 0 vers le pic 2 en respectant les règles du jeu.

Le système expert devra :

* représenter une situation du jeu ;
* vérifier si un déplacement est autorisé ;
* appliquer des règles de déplacement ;
* mémoriser les situations déjà étudiées afin d'éviter les boucles ;
* utiliser une métarègle pour choisir une règle lorsqu'il existe plusieurs possibilités ;
* poursuivre les déplacements jusqu'à atteindre la situation finale.

La résolution sera ensuite analysée en comparant le nombre de coups obtenu avec le nombre minimal de coups nécessaire.


In [3]:
# Import
import numpy as np

In [5]:
# Classe Jeu_Hanoi
class Jeu_Hanoi:
    def __init__(self):
        # Matrice représentant les 3 pics.
        # Les colonnes représentent les positions des disques,
        # avec la colonne 0 à la base.
        self.pic = np.zeros((3, 3), dtype=int)

        # Nombre de disques présents sur chaque pic.
        self.nombre_palet = np.zeros(3, dtype=int)

In [7]:
# Créer l'état initial
jeu = Jeu_Hanoi()

jeu.nombre_palet[0] = 3

jeu.pic[0, 0] = 3
jeu.pic[0, 1] = 2
jeu.pic[0, 2] = 1

print(jeu.pic)
print(jeu.nombre_palet)

[[3 2 1]
 [0 0 0]
 [0 0 0]]
[3 0 0]


In [9]:
# État final
jeu_final = Jeu_Hanoi()

jeu_final.nombre_palet[2] = 3

jeu_final.pic[2, 0] = 3
jeu_final.pic[2, 1] = 2
jeu_final.pic[2, 2] = 1

print(jeu_final.pic)
print(jeu_final.nombre_palet)

[[0 0 0]
 [0 0 0]
 [3 2 1]]
[0 0 3]


# ## 2. Vérification des conditions de déplacement

Avant d'effectuer un déplacement, le système expert doit vérifier que les conditions du jeu sont respectées.

La première condition consiste à vérifier que le pic de départ contient au moins un disque.


In [12]:
def pic_vide(indice_pic, jeu):
    return jeu.nombre_palet[indice_pic] == 0

In [14]:
print("Pic 0 vide :", pic_vide(0, jeu))
print("Pic 1 vide :", pic_vide(1, jeu))
print("Pic 2 vide :", pic_vide(2, jeu))

Pic 0 vide : False
Pic 1 vide : True
Pic 2 vide : True


## 3. Vérification d'un déplacement

Un déplacement d'un pic `indice_pic1` vers un pic `indice_pic2` est autorisé si :

1. le pic de départ contient au moins un disque ;
2. le pic d'arrivée est vide, ou son disque supérieur est plus grand que le disque que l'on souhaite déplacer.


In [17]:
# Vérifier si un déplacement est autorisé
def regle_jeu(indice_pic1, indice_pic2, jeu):
    # Le pic de départ doit contenir au moins un disque.
    if pic_vide(indice_pic1, jeu):
        return False

    # Disque situé au sommet du pic de départ.
    disque_depart = jeu.pic[
        indice_pic1,
        jeu.nombre_palet[indice_pic1] - 1
    ]

    # Si le pic d'arrivée est vide, le déplacement est autorisé.
    if pic_vide(indice_pic2, jeu):
        return True

    # Sinon, le disque supérieur du pic d'arrivée doit être plus grand.
    disque_arrivee = jeu.pic[
        indice_pic2,
        jeu.nombre_palet[indice_pic2] - 1
    ]

    return disque_arrivee > disque_depart

In [19]:
# Test 1 - Déplacement initial 0 → 1
print("Déplacement 0 -> 1 :", regle_jeu(0, 1, jeu))

Déplacement 0 -> 1 : True


In [21]:
# Test 2 - Déplacement 1 → 0
print("Déplacement 1 -> 0 :", regle_jeu(1, 0, jeu))

Déplacement 1 -> 0 : False


## 4. Effectuer un déplacement

Une fois qu'un déplacement est validé par `regle_jeu()`, il faut modifier la configuration du jeu.

Le disque supérieur du pic de départ est retiré puis placé au sommet du pic d'arrivée. Les nombres de disques présents sur les deux pics sont également mis à jour.


In [24]:
# Effectuer un déplacement
def effectue_deplacement(indice_pic1, indice_pic2, jeu):
    # Position du disque supérieur sur le pic de départ
    position_depart = jeu.nombre_palet[indice_pic1] - 1

    # Récupération du disque à déplacer
    disque = jeu.pic[indice_pic1, position_depart]

    # Suppression du disque du pic de départ
    jeu.pic[indice_pic1, position_depart] = 0
    jeu.nombre_palet[indice_pic1] -= 1

    # Position où placer le disque sur le pic d'arrivée
    position_arrivee = jeu.nombre_palet[indice_pic2]

    # Placement du disque
    jeu.pic[indice_pic2, position_arrivee] = disque
    jeu.nombre_palet[indice_pic2] += 1

In [26]:
jeu_test = Jeu_Hanoi()

jeu_test.nombre_palet[0] = 3
jeu_test.pic[0, 0] = 3
jeu_test.pic[0, 1] = 2
jeu_test.pic[0, 2] = 1

In [28]:
effectue_deplacement(0, 1, jeu_test)

print(jeu_test.pic)
print(jeu_test.nombre_palet)

[[3 2 0]
 [1 0 0]
 [0 0 0]]
[2 1 0]


In [30]:
print(regle_jeu(0, 1, jeu_test))

False


## 5. Encodage d'une situation

Pour éviter de revisiter indéfiniment les mêmes configurations, chaque situation du jeu doit être représentée par un nombre unique.

Le principe demandé dans le sujet est d'associer chaque disque à une puissance de 2 :

* disque 1 → `2^0`
* disque 2 → `2^1`
* disque 3 → `2^2`

Le numéro du pic est ensuite utilisé pour décaler la position associée au disque. Ainsi, chaque couple **(pic, disque)** correspond à un bit différent du nombre obtenu.


In [33]:
# Donner un identifiant unique à une situation
def nombre_situation(jeu):
    nombre = 0

    for b in range(3):
        for position in range(3):
            disque = jeu.pic[b, position]

            if disque != 0:
                # Chaque pic possède 3 positions de bits.
                # Le disque d occupe le bit (d - 1) du pic b.
                indice_bit = (disque - 1) + 3 * b
                nombre += 2 ** indice_bit

    return nombre

In [35]:
# Vérification sur l'état initial
nombre_initial = nombre_situation(jeu)

print("Identifiant de l'état initial :", nombre_initial)
print("Écriture binaire :", format(nombre_initial, "09b"))

Identifiant de l'état initial : 7
Écriture binaire : 000000111


In [37]:
# Vérification de l'état final
nombre_final = nombre_situation(jeu_final)

print("Identifiant de l'état final :", nombre_final)
print("Écriture binaire :", format(nombre_final, "09b"))

Identifiant de l'état final : 448
Écriture binaire : 111000000


## 6. Gestion des situations étudiées

Avant d'effectuer un déplacement, le système vérifie que la situation obtenue n'a pas déjà été étudiée.

La fonction simule le déplacement sur une copie du jeu afin de ne pas modifier l'état courant.


In [40]:
def situation_vue(indice_pic1, indice_pic2, jeu, situation_etudiee):
    # Copie du jeu courant
    jeu_test = Jeu_Hanoi()
    jeu_test.pic = jeu.pic.copy()
    jeu_test.nombre_palet = jeu.nombre_palet.copy()

    # Effectuer le déplacement sur la copie
    effectue_deplacement(indice_pic1, indice_pic2, jeu_test)

    # Identifier la nouvelle situation
    nouvelle_situation = nombre_situation(jeu_test)

    # True si la situation est nouvelle
    return nouvelle_situation not in situation_etudiee

In [42]:
situation_etudiee = []

print(situation_vue(0, 1, jeu, situation_etudiee))

True


In [44]:
jeu_test = Jeu_Hanoi()
jeu_test.pic = jeu.pic.copy()
jeu_test.nombre_palet = jeu.nombre_palet.copy()

effectue_deplacement(0, 1, jeu_test)

situation_etudiee.append(nombre_situation(jeu_test))

print(situation_vue(0, 1, jeu, situation_etudiee))

False


## 7. Base de règles

Le système expert utilise quatre règles de déplacement :

* **R1** : pic 0 → pic 1
* **R2** : pic 1 → pic 2
* **R3** : pic 2 → pic 1
* **R4** : pic 1 → pic 0

Une règle est applicable si le déplacement est légal et si la situation obtenue n'a pas déjà été étudiée.

La métarègle impose de choisir la règle de plus petit indice lorsqu'il existe plusieurs règles applicables.


In [47]:
# Définition des règles
regles = [
    (0, 1),  # R1
    (1, 2),  # R2
    (2, 1),  # R3
    (1, 0)   # R4
]

In [49]:
# Tester les règles applicables
situation_etudiee = [nombre_situation(jeu)]

regles_applicables = []

for indice, (pic_depart, pic_arrivee) in enumerate(regles, start=1):
    if regle_jeu(pic_depart, pic_arrivee, jeu):
        if situation_vue(
            pic_depart,
            pic_arrivee,
            jeu,
            situation_etudiee
        ):
            regles_applicables.append(indice)

print("Règles applicables :", regles_applicables)

Règles applicables : [1]


In [51]:
# Moteur d'inférence
def choisir_regle(jeu, situation_etudiee):
    for indice, (pic_depart, pic_arrivee) in enumerate(regles, start=1):

        if regle_jeu(pic_depart, pic_arrivee, jeu):

            if situation_vue(
                pic_depart,
                pic_arrivee,
                jeu,
                situation_etudiee
            ):
                return indice

    return None

In [53]:
# Résolution complète
jeu = Jeu_Hanoi()

jeu.nombre_palet[0] = 3
jeu.pic[0, 0] = 3
jeu.pic[0, 1] = 2
jeu.pic[0, 2] = 1

nombre_final = nombre_situation(jeu_final)

situation_etudiee = [nombre_situation(jeu)]

nbre_coup = 0
historique_regles = []

while nombre_situation(jeu) != nombre_final:

    regle = choisir_regle(jeu, situation_etudiee)

    if regle is None:
        print("Aucune règle applicable.")
        break

    pic_depart, pic_arrivee = regles[regle - 1]

    effectue_deplacement(pic_depart, pic_arrivee, jeu)

    nbre_coup += 1
    historique_regles.append(regle)

    situation_etudiee.append(nombre_situation(jeu))

    print(
        f"Coup {nbre_coup} : R{regle} "
        f"({pic_depart} → {pic_arrivee})"
    )

print("\nNombre de coups :", nbre_coup)
print("Règles utilisées :", historique_regles)
print("Situations étudiées :", len(situation_etudiee))
print("Situation finale atteinte :", nombre_situation(jeu) == nombre_final)

Coup 1 : R1 (0 → 1)
Coup 2 : R2 (1 → 2)
Coup 3 : R1 (0 → 1)
Coup 4 : R3 (2 → 1)
Coup 5 : R4 (1 → 0)
Coup 6 : R2 (1 → 2)
Coup 7 : R1 (0 → 1)
Coup 8 : R2 (1 → 2)
Coup 9 : R1 (0 → 1)
Coup 10 : R3 (2 → 1)
Coup 11 : R4 (1 → 0)
Coup 12 : R3 (2 → 1)
Coup 13 : R1 (0 → 1)
Coup 14 : R2 (1 → 2)
Coup 15 : R4 (1 → 0)
Coup 16 : R3 (2 → 1)
Coup 17 : R4 (1 → 0)
Coup 18 : R2 (1 → 2)
Coup 19 : R1 (0 → 1)
Coup 20 : R2 (1 → 2)
Coup 21 : R1 (0 → 1)
Coup 22 : R3 (2 → 1)
Coup 23 : R4 (1 → 0)
Coup 24 : R2 (1 → 2)
Coup 25 : R1 (0 → 1)
Coup 26 : R2 (1 → 2)

Nombre de coups : 26
Règles utilisées : [1, 2, 1, 3, 4, 2, 1, 2, 1, 3, 4, 3, 1, 2, 4, 3, 4, 2, 1, 2, 1, 3, 4, 2, 1, 2]
Situations étudiées : 27
Situation finale atteinte : True


In [55]:
print("Bilan de la résolution ")
print("Nombre de coups :", nbre_coup)
print("Nombre optimal :", 2**3 - 1)
print("Écart à l'optimal :", nbre_coup - (2**3 - 1))
print("Facteur par rapport à l'optimal :", nbre_coup / (2**3 - 1))

Bilan de la résolution 
Nombre de coups : 26
Nombre optimal : 7
Écart à l'optimal : 19
Facteur par rapport à l'optimal : 3.7142857142857144


## 8. Analyse de la première résolution

Le système expert atteint correctement la situation finale en **26 coups**.

Pour trois disques, le nombre minimal de déplacements est :

$$
2^3-1=7
$$

La stratégie utilisée par le système expert nécessite donc **19 coups supplémentaires** par rapport à l'optimum.

Cette différence s'explique par la stratégie de sélection des règles. La métarègle actuelle choisit simplement la règle applicable ayant le plus petit indice. Elle ne cherche pas à minimiser la distance restante jusqu'à l'état final.

Le système est donc capable de résoudre le problème, mais la stratégie de décision n'est pas optimale.


### Référence optimale

Pour les Tours de Hanoï classiques, le nombre minimal de déplacements pour \(n\) disques est :

$$
N_{\min}=2^n-1
$$

Pour \(n=3\) :

$$
N_{\min}=2^3-1=7
$$

Le système expert de base nécessite 26 coups. Son nombre de déplacements est donc environ **3,71 fois supérieur** à l'optimum.


## 9. Amélioration de la stratégie

La première version atteint l'état final, mais nécessite 26 coups. Le problème vient principalement de la métarègle qui sélectionne simplement la première règle applicable.

Pour améliorer le comportement, on ajoute les déplacements directs entre les pics 0 et 2 :

* R5 : pic 0 → pic 2
* R6 : pic 2 → pic 0

La stratégie améliorée donne également une priorité au déplacement du plus petit disque, en respectant son cycle de déplacement :

$$
0 \rightarrow 2 \rightarrow 1 \rightarrow 0
$$

Cette stratégie est adaptée au cas étudié de trois disques et permet de rapprocher la sélection des règles de la solution optimale.


In [60]:
regles_ameliorees = [
    (0, 1),  # R1
    (1, 2),  # R2
    (2, 1),  # R3
    (1, 0),  # R4
    (0, 2),  # R5
    (2, 0)   # R6
]

In [62]:
# Nouvelle sélection de règle
def choisir_regle_amelioree(jeu, numero_coup):
    # Position du plus petit disque
    position_petit = None

    for pic in range(3):
        for position in range(3):
            if jeu.pic[pic, position] == 1:
                position_petit = pic

    # Le plus petit disque suit le cycle 0 -> 2 -> 1 -> 0
    destination = {
        0: 2,
        2: 1,
        1: 0
    }

    # Déplacement du plus petit disque aux coups impairs
    if numero_coup % 2 == 1:
        pic_depart = position_petit
        pic_arrivee = destination[position_petit]

        for indice, (depart, arrivee) in enumerate(regles_ameliorees, start=1):
            if depart == pic_depart and arrivee == pic_arrivee:
                if regle_jeu(depart, arrivee, jeu):
                    return indice

    # Aux coups pairs, déplacer un autre disque
    for indice, (depart, arrivee) in enumerate(regles_ameliorees, start=1):
        if regle_jeu(depart, arrivee, jeu) and depart != position_petit:
            return indice

    return None

In [64]:
# Tester la nouvelle résolution
jeu = Jeu_Hanoi()

jeu.nombre_palet[0] = 3
jeu.pic[0, 0] = 3
jeu.pic[0, 1] = 2
jeu.pic[0, 2] = 1

nbre_coup_ameliore = 0
historique_ameliore = []

while nombre_situation(jeu) != nombre_final:

    nbre_coup_ameliore += 1

    regle = choisir_regle_amelioree(jeu, nbre_coup_ameliore)

    if regle is None:
        print("Aucune règle applicable.")
        break

    pic_depart, pic_arrivee = regles_ameliorees[regle - 1]

    effectue_deplacement(pic_depart, pic_arrivee, jeu)

    historique_ameliore.append(regle)

    print(
        f"Coup {nbre_coup_ameliore} : "
        f"R{regle} ({pic_depart} → {pic_arrivee})"
    )

print("\nNombre de coups :", nbre_coup_ameliore)
print("Règles utilisées :", historique_ameliore)
print("Situation finale atteinte :", nombre_situation(jeu) == nombre_final)

Coup 1 : R5 (0 → 2)
Coup 2 : R1 (0 → 1)
Coup 3 : R3 (2 → 1)
Coup 4 : R5 (0 → 2)
Coup 5 : R4 (1 → 0)
Coup 6 : R2 (1 → 2)
Coup 7 : R5 (0 → 2)

Nombre de coups : 7
Règles utilisées : [5, 1, 3, 5, 4, 2, 5]
Situation finale atteinte : True


In [66]:
# Comparaison finale
print("Comparaison ")
print("Stratégie initiale :", nbre_coup, "coups")
print("Stratégie améliorée :", nbre_coup_ameliore, "coups")
print("Optimum théorique :", 2**3 - 1, "coups")

Comparaison 
Stratégie initiale : 26 coups
Stratégie améliorée : 7 coups
Optimum théorique : 7 coups


In [68]:
# Tableau de comparaison
comparaison = {
    "Stratégie initiale": nbre_coup,
    "Stratégie améliorée": nbre_coup_ameliore,
    "Optimum théorique": 2**3 - 1
}

for strategie, coups in comparaison.items():
    print(f"{strategie:<25} : {coups} coups")

Stratégie initiale        : 26 coups
Stratégie améliorée       : 7 coups
Optimum théorique         : 7 coups


## 10. Analyse des résultats

La première stratégie permet de résoudre correctement le problème, mais elle nécessite 26 déplacements. Cette performance est nettement supérieure au minimum théorique de 7 déplacements.

La cause principale est la métarègle initiale, qui sélectionne la première règle applicable sans prendre en compte la progression vers l'état final. Le système est donc capable de trouver une solution, mais les décisions locales peuvent entraîner des déplacements supplémentaires.

Une seconde stratégie a été introduite en enrichissant la base de règles et en modifiant le choix des déplacements. Pour trois disques, cette stratégie permet d'atteindre l'état final en 7 déplacements.

Les résultats obtenus sont donc :

| Stratégie           | Nombre de coups |
| ------------------- | --------------: |
| Stratégie initiale  |              26 |
| Stratégie améliorée |               7 |
| Optimum théorique   |               7 |

La stratégie améliorée atteint ainsi l'optimum pour le problème étudié. Cette comparaison montre l'importance de la stratégie de sélection des règles dans un système expert : la présence de règles correctes ne garantit pas à elle seule une résolution efficace.


## 11. Conclusion

Ce projet a permis de construire un système expert à base de règles pour résoudre les Tours de Hanoï par chaînage avant.

Le système repose sur une représentation explicite des configurations, des règles de déplacement, une mémorisation des situations déjà étudiées et une métarègle permettant de sélectionner une règle applicable.

La première version a montré qu'un système expert peut atteindre l'objectif tout en produisant une solution non optimale. L'amélioration de la base de règles et de la stratégie de sélection a ensuite permis d'obtenir les 7 déplacements correspondant à l'optimum théorique pour trois disques.

L'expérience met ainsi en évidence le rôle essentiel de la stratégie d'inférence dans les performances d'un système expert : les règles définissent les possibilités d'action, tandis que leur organisation et leur priorité déterminent en grande partie la qualité de la résolution.
